In [94]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

In [95]:
# Carga de datos
df = pd.read_csv("data/WA_Fn-UseC_-HR-Employee-Attrition.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1470 non-null   int64
 1   Attrition                 1470 non-null   str  
 2   BusinessTravel            1470 non-null   str  
 3   DailyRate                 1470 non-null   int64
 4   Department                1470 non-null   str  
 5   DistanceFromHome          1470 non-null   int64
 6   Education                 1470 non-null   int64
 7   EducationField            1470 non-null   str  
 8   EmployeeCount             1470 non-null   int64
 9   EmployeeNumber            1470 non-null   int64
 10  EnvironmentSatisfaction   1470 non-null   int64
 11  Gender                    1470 non-null   str  
 12  HourlyRate                1470 non-null   int64
 13  JobInvolvement            1470 non-null   int64
 14  JobLevel                  1470 non-null   int64
 15

In [96]:
from sklearn.preprocessing import LabelEncoder

# Recodificación de la variable objetivo
le = LabelEncoder()
df["Attrition"] = le.fit_transform(df["Attrition"])

# Separación de variables categóricas y numéricas
cat_cols = df.select_dtypes(include=['object']).columns # Variables categóricas
df_encoded = pd.get_dummies(df[cat_cols], drop_first=True) # Variables categóricas codificadas
df_num = df.drop(columns=cat_cols) # Variables numéricas

# Dataset final
df_final = pd.concat([df_num, df_encoded], axis=1)
df_final.info()


<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 48 columns):
 #   Column                             Non-Null Count  Dtype
---  ------                             --------------  -----
 0   Age                                1470 non-null   int64
 1   Attrition                          1470 non-null   int64
 2   DailyRate                          1470 non-null   int64
 3   DistanceFromHome                   1470 non-null   int64
 4   Education                          1470 non-null   int64
 5   EmployeeCount                      1470 non-null   int64
 6   EmployeeNumber                     1470 non-null   int64
 7   EnvironmentSatisfaction            1470 non-null   int64
 8   HourlyRate                         1470 non-null   int64
 9   JobInvolvement                     1470 non-null   int64
 10  JobLevel                           1470 non-null   int64
 11  JobSatisfaction                    1470 non-null   int64
 12  MonthlyIncome                  

C:\Users\diego\AppData\Local\Temp\ipykernel_3932\1187089221.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns # Variables categóricas


In [97]:
# Frecuencias de la variable Attrition
df_final["Attrition"].value_counts(normalize=False)

Attrition
0    1233
1     237
Name: count, dtype: int64

In [98]:
# Selección variables dependientes (destino) e independientes (características)
X = df_final.drop(columns=["Attrition"])
y = df_final["Attrition"]

# Normalización de las variables independientes
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# División de datos en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=0)



### Modelo 1. Regresión logística

In [99]:
# Modelo de regresión logística
model = LogisticRegression()
model.fit(X_train, y_train)

# Predicciones
y_pred = model.predict(X_test)

# Probabilidades de pertenencia a cada clase
y_pred_prob = model.predict_proba(X_test)[:, 1]

# Evaluación del modelo
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)
curva_roc = roc_auc_score(y_test, y_pred_prob)

print("Matriz de confusión:")
print(conf_matrix)
print("Informe de clasificación:")
print(class_report)
print(f'Área bajo la curva ROC: {curva_roc:.2f}')

Matriz de confusión:
[[240   5]
 [ 29  20]]
Informe de clasificación:
              precision    recall  f1-score   support

           0       0.89      0.98      0.93       245
           1       0.80      0.41      0.54        49

    accuracy                           0.88       294
   macro avg       0.85      0.69      0.74       294
weighted avg       0.88      0.88      0.87       294

Área bajo la curva ROC: 0.85


### Modelo 2. Árbol de decisión

In [102]:
# Crear el modelo de árbol
model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=None,
    min_samples_split=2,
    random_state=0
)

# Entrenamiento
model.fit(X_train, y_train)

# Predicciones
y_pred = model.predict(X_test)

# Métricas
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nReporte completo:")
print(classification_report(y_test, y_pred))


Accuracy : 0.7891156462585034
Precision: 0.35555555555555557
Recall   : 0.32653061224489793
F1 Score : 0.3404255319148936

Reporte completo:
              precision    recall  f1-score   support

           0       0.87      0.88      0.87       245
           1       0.36      0.33      0.34        49

    accuracy                           0.79       294
   macro avg       0.61      0.60      0.61       294
weighted avg       0.78      0.79      0.79       294



### Modelo 3: K-vecionos mas cercanos (KNN)

In [101]:
# Escalar variables
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Modelo KNN
knn = KNeighborsClassifier(
    n_neighbors=5,      # puedes ajustar este hiperparámetro
    metric="minkowski", # distancia Euclidiana
    p=2
)

# Entrenamiento del modelo
knn.fit(X_train_scaled, y_train)

# Predicciones
y_pred = knn.predict(X_test_scaled)

# Métricas
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nReporte completo:")
print(classification_report(y_test, y_pred))


Accuracy : 0.8401360544217688
Precision: 0.5833333333333334
Recall   : 0.14285714285714285
F1 Score : 0.22950819672131148

Reporte completo:
              precision    recall  f1-score   support

           0       0.85      0.98      0.91       245
           1       0.58      0.14      0.23        49

    accuracy                           0.84       294
   macro avg       0.72      0.56      0.57       294
weighted avg       0.81      0.84      0.80       294

